# 11.31 — Safe & Constrained RL

Safe and constrained reinforcement learning asks an agent to maximize discounted reward while keeping discounted safety cost below a budget. In this lesson, the budget is not a vague warning label: it is a mathematical constraint inside a tiny constrained Markov decision process (CMDP), and we will build the value estimates, policy probabilities, Lagrangian penalty, and primal-dual updates from scratch with NumPy.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build safe RL one idea at a time. Run each cell in order and read the printed numbers: the toy environment is deliberately small so that reward, cost, discounting, constraints, and the Lagrangian are visible rather than hidden inside a library. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix so it never clashes with the exercises below.

In [ ]:
import numpy as np  # arrays, expectations, and small tabular RL calculations.
import matplotlib.pyplot as plt  # compact plots for reward/cost tradeoffs and learning curves.
np.random.seed(0)  # reproducibility for any sampled behavior.

### 1. A constrained MDP: reward and cost are two ledgers

A standard MDP has states, actions, transitions, and rewards. A **constrained** MDP adds a second signal: cost. The policy should seek high reward, but its expected discounted cost must stay below a budget. Here state `0` is a start state, state `1` is a risky high-payoff state, state `2` is a safe low-payoff state, and state `3` is terminal.

In [ ]:
states_w = ["start", "risky", "safe", "done"]  # four readable states.
actions_w = ["cautious", "bold"]  # two actions per nonterminal state.
gamma_w = 0.9  # discount factor: future reward/cost matters, but slightly less than now.
P_w = np.zeros((4, 2, 4))  # P[s, a, s'] transition probabilities.
R_w = np.zeros((4, 2, 4))  # reward earned on each transition.
C_w = np.zeros((4, 2, 4))  # safety cost earned on each transition.
print("P shape:", P_w.shape, "R shape:", R_w.shape, "C shape:", C_w.shape)

▶ What you'll see: all three tables have shape `(states, actions, next_states)`.

In [ ]:
P_w[0, 0, 2] = 1.0; R_w[0, 0, 2] = 1.0; C_w[0, 0, 2] = 0.1  # cautious start -> safe.
P_w[0, 1, 1] = 1.0; R_w[0, 1, 1] = 2.0; C_w[0, 1, 1] = 1.0  # bold start -> risky.
P_w[1, 0, 3] = 1.0; R_w[1, 0, 3] = 2.0; C_w[1, 0, 3] = 0.2  # brake from risky.
P_w[1, 1, 3] = 1.0; R_w[1, 1, 3] = 5.0; C_w[1, 1, 3] = 3.0  # push from risky.
P_w[2, 0, 3] = 1.0; R_w[2, 0, 3] = 2.0; C_w[2, 0, 3] = 0.1  # safe finish.
P_w[2, 1, 3] = 1.0; R_w[2, 1, 3] = 3.0; C_w[2, 1, 3] = 0.8  # faster safe finish.
P_w[3, :, 3] = 1.0  # terminal stays terminal.
print("start action rewards:", [R_w[0, a].sum() for a in range(2)])
print("start action costs:", [C_w[0, a].sum() for a in range(2)])
assert P_w[0, 0].sum() == 1.0 and P_w[0, 1].sum() == 1.0

▶ What you'll see: the bold action has larger immediate reward and larger immediate cost.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].bar(actions_w, [R_w[0, a].sum() for a in range(2)], color="seagreen")
ax[0].set_title("1: start reward")
ax[0].set_ylabel("immediate reward")
ax[1].bar(actions_w, [C_w[0, a].sum() for a in range(2)], color="indianred")
ax[1].set_title("1: start cost")
ax[1].set_ylabel("immediate cost")
plt.tight_layout(); plt.show()

▶ What you'll see: reward and cost point in the same tempting direction at the start — bold looks attractive but unsafe.

*Why it's done this way:* a CMDP separates utility from safety harm. If we collapsed cost into reward too early, we would hide the real requirement: maximize $J(\pi)$ while satisfying $C(\pi)\le d$. Keeping two ledgers lets us ask whether a policy is feasible before asking whether it is best among feasible policies.

### 2. Discounted reward and discounted cost returns

A trajectory has two discounted sums: reward return $G_R=\sum_t \gamma^t r_t$ and cost return $G_C=\sum_t \gamma^t c_t$. Discounting is applied to both ledgers because later benefit and later harm are consequences of today's action.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # worked reward stream from the source lesson.
costs_w = np.array([0.1, 0.0, 0.5])  # matching safety costs for the same three times.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
print("discount powers:", np.round(powers_w, 3))

▶ What you'll see: time 0 has weight 1, time 1 has 0.9, and time 2 has 0.81.

In [ ]:
G_reward_w = float(np.sum(powers_w * rewards_w))
G_cost_w = float(np.sum(powers_w * costs_w))
print("discounted reward:", round(G_reward_w, 3))
print("discounted cost:", round(G_cost_w, 3))
assert round(G_reward_w, 3) == 2.62
assert round(G_cost_w, 3) == 0.505

▶ What you'll see: the delayed reward of 2 counts as 1.62, giving total reward 2.62.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(powers_w * rewards_w, marker="o", label="discounted reward", color="seagreen")
plt.plot(powers_w * costs_w, marker="o", label="discounted cost", color="indianred")
plt.xlabel("time step"); plt.ylabel("discounted contribution")
plt.title("2: reward and cost returns use the same clock")
plt.legend(); plt.show()

▶ What you'll see: the final reward still matters, but discounting makes it smaller than its raw value.

*Why it's done this way:* $\gamma$ is the mathematical link between now and later. Safe RL cannot inspect only immediate cost, because a policy can take a cheap-looking action that opens a dangerous future. Discounting lets us compare trajectories of different lengths while keeping both reward and cost finite.

### 3. Evaluating a fixed policy by Bellman expectation

For a policy $\pi(a\mid s)$, the reward value and cost value obey two parallel Bellman equations:
$V_R(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)(R+\gamma V_R(s'))$ and similarly for $V_C$. We can solve them by repeated backups because the toy horizon is short.

In [ ]:
pi_w = np.array([[0.7, 0.3],   # at start: mostly cautious, sometimes bold.
                 [0.8, 0.2],   # if risky: mostly brake.
                 [0.6, 0.4],   # if safe: usually cautious finish.
                 [1.0, 0.0]])  # terminal action choice is irrelevant.
print("policy rows sum to:", pi_w.sum(axis=1))
assert np.allclose(pi_w.sum(axis=1), 1.0)

▶ What you'll see: every state has a valid probability distribution over actions.

In [ ]:
def eval_policy_w(P, R, C, pi, gamma, iters=50):
    Vr = np.zeros(P.shape[0])  # reward value for each state.
    Vc = np.zeros(P.shape[0])  # cost value for each state.
    for _ in range(iters):
        old_r, old_c = Vr.copy(), Vc.copy()
        for s in range(P.shape[0]):
            Vr[s] = sum(pi[s, a] * np.sum(P[s, a] * (R[s, a] + gamma * old_r)) for a in range(P.shape[1]))
            Vc[s] = sum(pi[s, a] * np.sum(P[s, a] * (C[s, a] + gamma * old_c)) for a in range(P.shape[1]))
    return Vr, Vc
Vr_w, Vc_w = eval_policy_w(P_w, R_w, C_w, pi_w, gamma_w)
print("reward values:", np.round(Vr_w, 3))
print("cost values:", np.round(Vc_w, 3))

▶ What you'll see: state 0 has both the expected discounted reward and expected discounted cost of starting under this policy.

In [ ]:
budget_w = 0.9
print("J(pi) from start:", round(Vr_w[0], 3))
print("C(pi) from start:", round(Vc_w[0], 3), "budget:", budget_w)
print("feasible?", Vc_w[0] <= budget_w)
assert round(Vr_w[0], 3) == 3.514
assert round(Vc_w[0], 3) == 0.815

▶ What you'll see: the mixed policy is just under the 0.9 cost budget.

*Why it's done this way:* policy evaluation averages over both action randomness and transition randomness. The Bellman backup is not magic; it is just conditional expectation: immediate ledger entry plus discounted value of the next state, weighted by how likely each action and transition is.

### 4. Feasible policies form a reward–cost frontier

The constraint $C(\pi)\le d$ cuts the policy space into feasible and infeasible regions. To see that geometry, we sweep the probability of choosing `bold` at the start and keep the later-state policy fixed.

In [ ]:
bold_grid_w = np.linspace(0, 1, 21)  # possible probabilities for bold at the start.
frontier_reward_w, frontier_cost_w = [], []
for p_bold_w in bold_grid_w:
    pi_tmp_w = pi_w.copy()
    pi_tmp_w[0] = [1 - p_bold_w, p_bold_w]
    vr_tmp_w, vc_tmp_w = eval_policy_w(P_w, R_w, C_w, pi_tmp_w, gamma_w)
    frontier_reward_w.append(vr_tmp_w[0]); frontier_cost_w.append(vc_tmp_w[0])
frontier_reward_w = np.array(frontier_reward_w); frontier_cost_w = np.array(frontier_cost_w)
print("cost range:", round(frontier_cost_w.min(), 3), "to", round(frontier_cost_w.max(), 3))

▶ What you'll see: more bold-start probability smoothly increases both reward and cost.

In [ ]:
feasible_w = frontier_cost_w <= budget_w
best_idx_w = int(np.argmax(np.where(feasible_w, frontier_reward_w, -np.inf)))
print("best feasible bold probability:", round(float(bold_grid_w[best_idx_w]), 2))
print("best feasible reward/cost:", round(frontier_reward_w[best_idx_w], 3), round(frontier_cost_w[best_idx_w], 3))
assert round(float(bold_grid_w[best_idx_w]), 2) == 0.35

▶ What you'll see: the best feasible grid policy uses bold with probability 0.35 at the start.

In [ ]:
plt.figure(figsize=(5, 3.4))
plt.scatter(frontier_cost_w, frontier_reward_w, c=feasible_w, cmap="coolwarm", s=45)
plt.axvline(budget_w, color="black", linestyle="--", label="cost budget")
plt.scatter([frontier_cost_w[best_idx_w]], [frontier_reward_w[best_idx_w]], color="gold", edgecolor="black", s=100, label="best feasible")
plt.xlabel("discounted cost C(pi)"); plt.ylabel("discounted reward J(pi)")
plt.title("4: reward-vs-cost frontier")
plt.legend(); plt.show()

▶ What you'll see: points to the right of the dashed budget line may have higher reward, but they are not allowed.

*Why it's done this way:* constrained optimization is not the same as adding a fixed penalty by instinct. The frontier shows the real tradeoff: among feasible policies, choose the highest reward; infeasible high-reward policies are outside the problem definition no matter how exciting their reward looks.

### 5. The Lagrangian turns a hard budget into a priced violation

A common safe-RL trick is the Lagrangian $L(\pi,\lambda)=J(\pi)-\lambda(C(\pi)-d)$ with $\lambda\ge0$. If cost exceeds the budget, a larger $\lambda$ makes that policy less attractive; if cost is under budget, the penalty relaxes.

In [ ]:
lambdas_w = np.array([0.0, 1.0, 3.0])  # three possible safety prices.
lag_values_w = {}
for lam_w in lambdas_w:
    lag_values_w[lam_w] = frontier_reward_w - lam_w * (frontier_cost_w - budget_w)
    chosen_w = int(np.argmax(lag_values_w[lam_w]))
    print("lambda", lam_w, "chooses bold p=", round(float(bold_grid_w[chosen_w]), 2),
          "cost=", round(frontier_cost_w[chosen_w], 3))

▶ What you'll see: when lambda is zero, the optimizer chases reward; when lambda is larger, costly policies lose appeal.

In [ ]:
plt.figure(figsize=(5, 3.2))
for lam_w in lambdas_w:
    plt.plot(bold_grid_w, lag_values_w[lam_w], marker="o", label=f"lambda={lam_w:g}")
plt.xlabel("start bold probability"); plt.ylabel("L(pi, lambda)")
plt.title("5: safety price reshapes the objective")
plt.legend(); plt.show()

▶ What you'll see: increasing the safety price shifts the peak of the objective toward safer behavior.

*Why it's done this way:* the multiplier $\lambda$ is a learned exchange rate between reward and constraint violation. It is not an arbitrary constant reward hack: dual ascent increases it exactly when the current policy spends more cost than the allowed budget.

### 6. Primal-dual learning: update policy and safety price together

In tabular safe RL, one simple approach is primal-dual learning. The **primal** step improves the policy under the current Lagrangian reward $R-\lambda C$; the **dual** step raises $\lambda$ when measured cost is above budget and lowers it when cost is below budget.

In [ ]:
logit_w = 0.0  # one learnable number controlling bold probability at the start.
lam_pd_w = 0.0  # safety price starts at zero.
history_w = []  # store (p_bold, reward, cost, lambda).
eta_pi_w, eta_lam_w = 0.8, 0.8  # small toy learning rates.
print("initial bold probability:", round(1 / (1 + np.exp(-logit_w)), 3))

▶ What you'll see: a zero logit means a 50/50 start policy before safety learning.

In [ ]:
for step_w in range(35):
    p_w = 1 / (1 + np.exp(-logit_w))  # sigmoid turns the logit into bold probability.
    pi_step_w = pi_w.copy(); pi_step_w[0] = [1 - p_w, p_w]
    vr_step_w, vc_step_w = eval_policy_w(P_w, R_w, C_w, pi_step_w, gamma_w)
    history_w.append([p_w, vr_step_w[0], vc_step_w[0], lam_pd_w])
    # Finite-difference gradient of the Lagrangian objective with respect to the logit.
    eps_w = 1e-3
    for sign_w in [1, -1]:
        p_eps_w = 1 / (1 + np.exp(-(logit_w + sign_w * eps_w)))
        pi_eps_w = pi_w.copy(); pi_eps_w[0] = [1 - p_eps_w, p_eps_w]
        vr_eps_w, vc_eps_w = eval_policy_w(P_w, R_w, C_w, pi_eps_w, gamma_w)
        if sign_w == 1:
            plus_w = vr_eps_w[0] - lam_pd_w * (vc_eps_w[0] - budget_w)
        else:
            minus_w = vr_eps_w[0] - lam_pd_w * (vc_eps_w[0] - budget_w)
    grad_logit_w = (plus_w - minus_w) / (2 * eps_w)
    logit_w += eta_pi_w * grad_logit_w
    lam_pd_w = max(0.0, lam_pd_w + eta_lam_w * (vc_step_w[0] - budget_w))
history_w = np.array(history_w)
print("final p/reward/cost/lambda:", np.round(history_w[-1], 3))

▶ What you'll see: the cost moves toward the budget while lambda rises only when the policy overspends cost.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(history_w[:, 1], label="reward", color="seagreen")
ax[0].plot(history_w[:, 2], label="cost", color="indianred")
ax[0].axhline(budget_w, color="black", linestyle="--", label="budget")
ax[0].set_title("6: reward vs cost during learning"); ax[0].legend()
ax[1].plot(history_w[:, 0], label="p(bold)", color="purple")
ax[1].plot(history_w[:, 3], label="lambda", color="gray")
ax[1].set_title("6: policy and safety price"); ax[1].legend()
plt.tight_layout(); plt.show()

▶ What you'll see: reward and cost move together at first; the learned multiplier then pushes the policy back toward the cost line.

*Why it's done this way:* the primal update asks, “given the current safety price, what behavior pays best?” The dual update asks, “did that behavior exceed the budget?” Their feedback loop is the simplest numerical version of constrained optimization.

### 7. Bootstrapping and policy support are safety pitfalls

Safe RL still inherits ordinary RL pitfalls. A one-step target can amplify a bad value estimate, and an offline dataset cannot reliably evaluate actions it almost never sampled. Safety makes these issues sharper because an error can hide true cost.

In [ ]:
r_boot_w, next_v_w, q_old_w, alpha_w = 1.0, 0.8, 0.4, 0.5
y_boot_w = r_boot_w + gamma_w * next_v_w
q_new_w = q_old_w + alpha_w * (y_boot_w - q_old_w)
print("bootstrap target:", round(y_boot_w, 3))
print("updated Q:", round(q_new_w, 3))
assert round(y_boot_w, 3) == 1.72 and round(q_new_w, 3) == 1.06

▶ What you'll see: the update moves halfway toward a target that itself depends on a current estimate.

In [ ]:
behavior_counts_w = np.array([95, 5])  # offline data: cautious is common, bold is rare.
eval_policy_w_support = np.array([0.2, 0.8])  # proposed policy mostly wants bold.
coverage_w = behavior_counts_w / behavior_counts_w.sum()
print("behavior coverage:", np.round(coverage_w, 2))
print("eval policy mass:", eval_policy_w_support)
print("unsupported bold mass warning:", eval_policy_w_support[1] > coverage_w[1] * 5)

▶ What you'll see: the policy being evaluated puts most mass on an action with very little data support.

In [ ]:
plt.figure(figsize=(5, 3))
width_w = 0.35
x_w = np.arange(2)
plt.bar(x_w - width_w/2, coverage_w, width_w, label="dataset frequency", color="gray")
plt.bar(x_w + width_w/2, eval_policy_w_support, width_w, label="eval policy", color="orange")
plt.xticks(x_w, actions_w); plt.ylabel("probability")
plt.title("7: off-policy support mismatch")
plt.legend(); plt.show()

▶ What you'll see: the policy wants to use bold far more often than the data observed it, so its cost estimate is fragile.

*Why it's done this way:* bootstrapping trades variance for bias by trusting the learner's own estimate, and off-policy evaluation trades direct evidence for extrapolation. In safety work, both require extra checks because underestimated cost can falsely certify an unsafe policy.

## 🛠️ Setup

In [ ]:
import numpy as np # Load NumPy for arrays, expectations, tabular policies, and numerical assertions.
import matplotlib.pyplot as plt # Load Matplotlib for reward-cost frontiers, learning curves, and heatmaps.
np.random.seed(0) # Make every stochastic example reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a discounted reward return

**Goal.** Turn a short reward stream into a discounted return, because safe RL still optimizes future consequence rather than immediate reward alone. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0]) # Define the worked reward stream from the lesson text.
gamma_b1 = 0.9 # Choose the discount factor that weights future rewards.
powers_b1 = gamma_b1 ** np.arange(len(rewards_b1)) # Compute [1, gamma, gamma^2] for the three rewards.
print("discount powers:", np.round(powers_b1, 3)) # Inspect each time step's weight.

▶ What you'll see: the third reward is weighted by 0.81 because it arrives two steps later.

In [ ]:
return_b1 = float(np.sum(powers_b1 * rewards_b1)) # Sum discounted rewards into one return.
print("G:", round(return_b1, 3)) # Inspect the scalar objective contribution.
assert round(return_b1, 3) == 2.62 # Verify 1 + 0.9*0 + 0.9^2*2.
plt.figure(figsize=(4, 3)) # Create a compact contribution chart.
plt.bar(["t0", "t1", "t2"], powers_b1 * rewards_b1, color="seagreen") # Show each discounted reward term.
plt.title("Basic 1: discounted reward terms") # Title the chart.
plt.ylabel("gamma^t r_t") # Label the contribution scale.
plt.show() # Display the chart.

▶ What you'll see: the delayed reward contributes 1.62, not the raw value 2.

👀 Takeaway: return is a discounted sum, so delayed rewards matter but are not treated as free.

### Basic 2 — Compute a discounted cost return

**Goal.** Apply the same return logic to safety costs, because constraints are written on expected cumulative cost. We build it in 2 steps.

In [ ]:
costs_b2 = np.array([0.1, 0.0, 0.5]) # Define a short stream of safety costs.
gamma_b2 = 0.9 # Use the same discount clock as the reward ledger.
powers_b2 = gamma_b2 ** np.arange(len(costs_b2)) # Compute time weights for costs.
print("discounted cost terms:", np.round(powers_b2 * costs_b2, 3)) # Inspect each cost contribution.

▶ What you'll see: the last cost is discounted to 0.405.

In [ ]:
cost_return_b2 = float(np.sum(powers_b2 * costs_b2)) # Sum discounted costs into C(pi) for this trajectory.
print("discounted cost:", round(cost_return_b2, 3)) # Inspect the cost ledger total.
assert round(cost_return_b2, 3) == 0.505 # Verify 0.1 + 0 + 0.81*0.5.
plt.figure(figsize=(4, 3)) # Create a compact cost contribution chart.
plt.bar(["t0", "t1", "t2"], powers_b2 * costs_b2, color="indianred") # Plot discounted safety costs.
plt.title("Basic 2: discounted cost terms") # Title the chart.
plt.ylabel("gamma^t c_t") # Label the cost scale.
plt.show() # Display the chart.

▶ What you'll see: cumulative cost is a separate scalar that can be compared with a budget.

👀 Takeaway: safe RL constrains discounted cost, not just one-step cost.

### Basic 3 — Check a cost budget

**Goal.** Decide whether a trajectory or policy is feasible, because constrained RL only optimizes among policies whose cost is below the budget. We build it in 2 steps.

In [ ]:
C_b3 = 0.505 # Use the discounted cost computed in Basic 2.
budget_b3 = 0.6 # Set a safety budget for the example.
slack_b3 = budget_b3 - C_b3 # Positive slack means the constraint is satisfied.
print("cost:", C_b3, "budget:", budget_b3, "slack:", round(slack_b3, 3)) # Inspect feasibility ingredients.

▶ What you'll see: cost is below the budget, so slack is positive.

In [ ]:
feasible_b3 = C_b3 <= budget_b3 # Convert the inequality C(pi) <= d into a boolean check.
print("feasible?", feasible_b3) # Inspect whether the policy is allowed.
assert feasible_b3 # Verify this example satisfies the constraint.
plt.figure(figsize=(4, 3)) # Create a simple budget chart.
plt.bar(["cost", "budget"], [C_b3, budget_b3], color=["indianred", "gray"]) # Compare cost with allowed limit.
plt.title("Basic 3: feasibility check") # Title the chart.
plt.ylabel("discounted cost") # Label the y-axis.
plt.show() # Display the chart.

▶ What you'll see: the cost bar sits below the budget bar.

👀 Takeaway: feasibility is the inequality $C(\pi)\le d$, not a subjective label.

### Basic 4 — Softmax policy probabilities

**Goal.** Convert logits into action probabilities, because policy-gradient safe RL often changes logits rather than actions directly. We build it in 3 steps.

In [ ]:
logits_b4 = np.array([1.0, 0.0]) # Define two action preferences before normalization.
shifted_b4 = logits_b4 - np.max(logits_b4) # Shift logits for numerical stability without changing softmax.
exp_b4 = np.exp(shifted_b4) # Exponentiate shifted logits into positive weights.
print("exp weights:", np.round(exp_b4, 3)) # Inspect unnormalized action weights.

▶ What you'll see: the first action receives the larger positive weight.

In [ ]:
pi_b4 = exp_b4 / exp_b4.sum() # Normalize weights into probabilities that sum to one.
print("policy probabilities:", np.round(pi_b4, 3)) # Inspect pi(a|s) for two actions.
assert np.allclose(np.round(pi_b4, 3), np.array([0.731, 0.269])) # Verify the source lesson softmax numbers.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact policy chart.
plt.bar(["a0", "a1"], pi_b4, color="steelblue") # Show action probabilities after softmax.
plt.ylim(0, 1) # Keep the probability scale visible.
plt.title("Basic 4: softmax policy") # Title the chart.
plt.ylabel("probability") # Label the y-axis.
plt.show() # Display the chart.

▶ What you'll see: action 0 has probability about 0.731 and action 1 about 0.269.

👀 Takeaway: logits become probabilities, and probabilities determine expected reward and cost.

### Basic 5 — Expected reward under a policy

**Goal.** Average action rewards using policy probabilities, because a stochastic policy's objective is an expectation. We build it in 2 steps.

In [ ]:
pi_b5 = np.array([0.731, 0.269]) # Use the rounded softmax probabilities from Basic 4.
reward_by_action_b5 = np.array([2.0, 0.0]) # Define one-step rewards for two actions.
weighted_reward_b5 = pi_b5 * reward_by_action_b5 # Compute each action's expected contribution.
print("weighted rewards:", np.round(weighted_reward_b5, 3)) # Inspect probability times reward.

▶ What you'll see: only action 0 contributes because action 1's reward is zero.

In [ ]:
expected_reward_b5 = float(np.sum(weighted_reward_b5)) # Sum action contributions into expected reward.
print("expected reward:", round(expected_reward_b5, 3)) # Inspect E_pi[R].
assert round(expected_reward_b5, 3) == 1.462 # Verify 0.731*2.
plt.figure(figsize=(4, 3)) # Create a contribution chart.
plt.bar(["a0", "a1"], weighted_reward_b5, color="seagreen") # Plot each action's expected reward contribution.
plt.title("Basic 5: policy-weighted reward") # Title the chart.
plt.ylabel("pi(a) R(a)") # Label contribution scale.
plt.show() # Display the chart.

▶ What you'll see: the expected reward is 1.462, the probability-weighted average.

👀 Takeaway: stochastic policies optimize expectations, not the reward of only the most likely action.

### Basic 6 — Expected cost under a policy

**Goal.** Average action costs using the same policy probabilities, because a policy can be high reward and high cost at the same time. We build it in 2 steps.

In [ ]:
pi_b6 = np.array([0.731, 0.269]) # Reuse a two-action stochastic policy.
cost_by_action_b6 = np.array([0.2, 2.0]) # Define one safe action and one risky action.
weighted_cost_b6 = pi_b6 * cost_by_action_b6 # Compute each action's expected cost contribution.
print("weighted costs:", np.round(weighted_cost_b6, 3)) # Inspect probability times cost.

▶ What you'll see: the low-probability risky action still contributes visible expected cost.

In [ ]:
expected_cost_b6 = float(np.sum(weighted_cost_b6)) # Sum into expected one-step cost.
print("expected cost:", round(expected_cost_b6, 3)) # Inspect E_pi[C].
assert round(expected_cost_b6, 3) == 0.684 # Verify 0.731*0.2 + 0.269*2.
plt.figure(figsize=(4, 3)) # Create a compact cost chart.
plt.bar(["a0", "a1"], weighted_cost_b6, color="indianred") # Plot each action's expected cost contribution.
plt.title("Basic 6: policy-weighted cost") # Title the chart.
plt.ylabel("pi(a) cost(a)") # Label contribution scale.
plt.show() # Display the chart.

▶ What you'll see: even a 0.269 chance of the risky action dominates the expected cost.

👀 Takeaway: safety constraints care about probability mass on costly actions, not just the chosen argmax.

### Basic 7 — One Bellman target

**Goal.** Build the one-step target $y=r+\gamma V(s')$, because tabular RL updates often bootstrap from the next state estimate. We build it in 2 steps.

In [ ]:
r_b7 = 1.0 # Observed immediate reward.
next_v_b7 = 0.8 # Current estimate of the next state's value.
gamma_b7 = 0.9 # Discount applied to that next value.
y_b7 = r_b7 + gamma_b7 * next_v_b7 # Compute the bootstrap target.
print("target y:", round(y_b7, 3)) # Inspect r + gamma V(s').
assert round(y_b7, 3) == 1.72 # Verify the source lesson target.

▶ What you'll see: the target is 1.72, larger than immediate reward because the next state is valuable.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a target breakdown chart.
plt.bar(["r", "gamma V(next)", "target"], [r_b7, gamma_b7 * next_v_b7, y_b7], color=["green", "orange", "gray"]) # Compare target pieces.
plt.title("Basic 7: Bellman target") # Title the chart.
plt.xticks(rotation=15) # Rotate labels for readability.
plt.show() # Display the chart.

▶ What you'll see: the target combines immediate reward and discounted estimated future value.

👀 Takeaway: bootstrapping learns from partial experience plus the learner's current estimate.

### Basic 8 — One Q-learning update

**Goal.** Move an old action-value estimate toward a Bellman target, because learning rates prevent one sample from overwriting the table. We build it in 2 steps.

In [ ]:
q_old_b8 = 0.4 # Current action-value estimate.
y_b8 = 1.72 # Bootstrap target from Basic 7.
alpha_b8 = 0.5 # Learning rate: move halfway to the target.
td_error_b8 = y_b8 - q_old_b8 # Compute the temporal-difference error.
print("TD error:", round(td_error_b8, 3)) # Inspect the correction signal.

▶ What you'll see: the target is above the old estimate, so the TD error is positive.

In [ ]:
q_new_b8 = q_old_b8 + alpha_b8 * td_error_b8 # Update Q toward the target.
print("new Q:", round(q_new_b8, 3)) # Inspect the updated action value.
assert round(q_new_b8, 3) == 1.06 # Verify the source lesson update.
plt.figure(figsize=(4, 3)) # Create a before/after chart.
plt.bar(["old Q", "new Q", "target"], [q_old_b8, q_new_b8, y_b8], color=["gray", "teal", "black"]) # Compare old, updated, and target values.
plt.title("Basic 8: halfway TD update") # Title the chart.
plt.ylabel("value") # Label the y-axis.
plt.show() # Display the chart.

▶ What you'll see: the estimate moves from 0.4 to 1.06, halfway toward 1.72.

👀 Takeaway: a learning rate controls how strongly one transition changes an estimate.

### Basic 9 — UCB exploration pressure

**Goal.** Add an uncertainty bonus to a mean reward estimate, because exploration is valuable when an action has been sampled few times. We build it in 2 steps.

In [ ]:
mean_b9 = 0.55 # Current empirical mean reward for an action.
time_b9 = 20 # Total decision count so far.
count_b9 = 5 # Number of times this action was sampled.
c_b9 = 1.0 # Exploration coefficient.
bonus_b9 = c_b9 * np.sqrt(2 * np.log(time_b9) / count_b9) # Compute the UCB bonus.
print("bonus:", round(bonus_b9, 3)) # Inspect the uncertainty term.

▶ What you'll see: the exploration bonus is about 1.095.

In [ ]:
ucb_b9 = mean_b9 + bonus_b9 # Add optimism to the observed mean.
print("UCB index:", round(ucb_b9, 3)) # Inspect the action index used for selection.
assert round(ucb_b9, 3) == 1.645 # Verify the source lesson number.
plt.figure(figsize=(4, 3)) # Create a decomposition chart.
plt.bar(["mean", "bonus", "UCB"], [mean_b9, bonus_b9, ucb_b9], color=["gray", "orange", "purple"]) # Compare mean, uncertainty, and index.
plt.title("Basic 9: exploration bonus") # Title the chart.
plt.ylabel("score") # Label the score scale.
plt.show() # Display the chart.

▶ What you'll see: the index is much larger than the mean because the action is uncertain.

👀 Takeaway: exploration bonuses deliberately pay for information, not just immediate reward.

### Basic 10 — Lagrangian penalty for one policy

**Goal.** Compute $J-\lambda(C-d)$ for a candidate policy, because the multiplier prices budget violation. We build it in 3 steps.

In [ ]:
J_b10 = 3.5 # Candidate expected discounted reward.
C_b10 = 1.2 # Candidate expected discounted cost.
d_b10 = 0.9 # Safety budget.
lam_b10 = 2.0 # Current Lagrange multiplier.
violation_b10 = C_b10 - d_b10 # Positive means the policy overspends cost.
print("violation:", round(violation_b10, 3)) # Inspect C - d.

▶ What you'll see: the policy exceeds the budget by 0.3.

In [ ]:
lagrangian_b10 = J_b10 - lam_b10 * violation_b10 # Penalize reward by the priced violation.
print("Lagrangian value:", round(lagrangian_b10, 3)) # Inspect the safety-priced objective.
assert round(lagrangian_b10, 3) == 2.9 # Verify 3.5 - 2*0.3.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact objective chart.
plt.bar(["reward J", "penalty", "L"], [J_b10, lam_b10 * violation_b10, lagrangian_b10], color=["seagreen", "indianred", "gray"]) # Compare reward, penalty, and objective.
plt.title("Basic 10: priced constraint violation") # Title the chart.
plt.ylabel("value") # Label the value scale.
plt.show() # Display the chart.

▶ What you'll see: the safety penalty reduces the objective from 3.5 to 2.9.

👀 Takeaway: the Lagrangian lowers the score of policies that exceed the cost budget.

## 🟡 Easy

### Easy 1 — Evaluate two deterministic policies

**Goal.** Compare always-cautious and always-bold policies in a tiny CMDP, because safe RL chooses among policies using both reward and cost. We build it in 4 steps.

In [ ]:
P_e1 = np.zeros((3, 2, 3)) # Build a three-state CMDP: start, mid, done.
R_e1 = np.zeros((3, 2, 3)) # Reward table for transitions.
C_e1 = np.zeros((3, 2, 3)) # Cost table for transitions.
gamma_e1 = 0.9 # Discount factor.
P_e1[0, 0, 1] = 1; R_e1[0, 0, 1] = 1.0; C_e1[0, 0, 1] = 0.1 # cautious start.
P_e1[0, 1, 1] = 1; R_e1[0, 1, 1] = 2.0; C_e1[0, 1, 1] = 1.0 # bold start.
P_e1[1, 0, 2] = 1; R_e1[1, 0, 2] = 2.0; C_e1[1, 0, 2] = 0.1 # cautious finish.
P_e1[1, 1, 2] = 1; R_e1[1, 1, 2] = 4.0; C_e1[1, 1, 2] = 2.0 # bold finish.
P_e1[2, :, 2] = 1 # done is absorbing.
print("environment ready with states/actions:", P_e1.shape[:2]) # Inspect dimensions.

▶ What you'll see: a small CMDP with two decision states and two actions.

In [ ]:
def rollout_eval_e1(actions):
    s_e1 = 0; G_e1 = 0.0; K_e1 = 0.0 # Track state, reward return, and cost return.
    for t_e1, a_e1 in enumerate(actions):
        sp_e1 = int(np.argmax(P_e1[s_e1, a_e1])) # Deterministic next state.
        G_e1 += (gamma_e1 ** t_e1) * R_e1[s_e1, a_e1, sp_e1] # Add discounted reward.
        K_e1 += (gamma_e1 ** t_e1) * C_e1[s_e1, a_e1, sp_e1] # Add discounted cost.
        s_e1 = sp_e1 # Move to next state.
    return G_e1, K_e1
cautious_e1 = rollout_eval_e1([0, 0]) # Evaluate cautious then cautious.
bold_e1 = rollout_eval_e1([1, 1]) # Evaluate bold then bold.
print("cautious reward/cost:", np.round(cautious_e1, 3)) # Inspect safe policy.
print("bold reward/cost:", np.round(bold_e1, 3)) # Inspect risky policy.

In [ ]:
budget_e1 = 1.0 # Set the discounted cost budget.
print("cautious feasible?", cautious_e1[1] <= budget_e1) # Check cautious feasibility.
print("bold feasible?", bold_e1[1] <= budget_e1) # Check bold feasibility.
assert round(cautious_e1[0], 3) == 2.8 and round(cautious_e1[1], 3) == 0.19 # Verify cautious numbers.
assert round(bold_e1[0], 3) == 5.6 and round(bold_e1[1], 3) == 2.8 # Verify bold numbers.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a reward-cost comparison.
plt.scatter([cautious_e1[1], bold_e1[1]], [cautious_e1[0], bold_e1[0]], s=90, color=["seagreen", "indianred"]) # Plot cost on x and reward on y.
plt.axvline(budget_e1, color="black", linestyle="--", label="budget") # Draw budget line.
plt.text(cautious_e1[1], cautious_e1[0], " cautious") # Label safe policy.
plt.text(bold_e1[1], bold_e1[0], " bold") # Label risky policy.
plt.xlabel("discounted cost"); plt.ylabel("discounted reward") # Label axes.
plt.title("Easy 1: deterministic policies") # Title the plot.
plt.legend(); plt.show() # Display the plot.

▶ What you'll see: bold has higher reward but lies far beyond the cost budget.

👀 Takeaway: constrained RL rejects an infeasible high-reward policy rather than celebrating it.

### Easy 2 — Sweep a stochastic safety frontier

**Goal.** Sweep the probability of a bold action, because stochastic policies can interpolate between safe low reward and risky high reward. We build it in 4 steps.

In [ ]:
p_grid_e2 = np.linspace(0, 1, 11) # Candidate probabilities of choosing bold at both decision states.
gamma_e2 = 0.9 # Discount factor for two-step returns.
reward_e2 = [] # Store expected discounted rewards.
cost_e2 = [] # Store expected discounted costs.
print("grid:", np.round(p_grid_e2, 2)) # Inspect probabilities to be evaluated.

▶ What you'll see: probabilities from always cautious to always bold.

In [ ]:
for p_e2 in p_grid_e2:
    J_e2 = (1 - p_e2) * 1.0 + p_e2 * 2.0 + gamma_e2 * ((1 - p_e2) * 2.0 + p_e2 * 4.0) # Expected reward at two states.
    K_e2 = (1 - p_e2) * 0.1 + p_e2 * 1.0 + gamma_e2 * ((1 - p_e2) * 0.1 + p_e2 * 2.0) # Expected cost at two states.
    reward_e2.append(J_e2); cost_e2.append(K_e2) # Save frontier point.
reward_e2 = np.array(reward_e2); cost_e2 = np.array(cost_e2) # Convert to arrays for filtering.
print("first/last reward:", round(reward_e2[0], 3), round(reward_e2[-1], 3)) # Inspect endpoints.
print("first/last cost:", round(cost_e2[0], 3), round(cost_e2[-1], 3)) # Inspect endpoints.

In [ ]:
budget_e2 = 1.0 # Safety budget.
feasible_e2 = cost_e2 <= budget_e2 # Boolean feasible mask.
best_e2 = int(np.argmax(np.where(feasible_e2, reward_e2, -np.inf))) # Best reward among feasible points.
print("best feasible p:", round(float(p_grid_e2[best_e2]), 2)) # Inspect selected probability.
assert round(float(p_grid_e2[best_e2]), 2) == 0.3 # Verify the grid selection.

In [ ]:
plt.figure(figsize=(5, 3)) # Create frontier plot.
plt.scatter(cost_e2, reward_e2, c=feasible_e2, cmap="coolwarm", s=60) # Color feasible vs infeasible policies.
plt.axvline(budget_e2, color="black", linestyle="--") # Draw the budget.
plt.scatter(cost_e2[best_e2], reward_e2[best_e2], s=120, color="gold", edgecolor="black") # Mark best feasible policy.
plt.xlabel("C(pi)"); plt.ylabel("J(pi)") # Label reward-cost axes.
plt.title("Easy 2: stochastic reward-cost frontier") # Title the plot.
plt.show() # Display the frontier.

▶ What you'll see: the frontier slopes upward, and the best feasible grid point is just left of the budget.

👀 Takeaway: a cost budget turns policy search into a constrained frontier problem.

### Easy 3 — Choose actions with a Lagrangian value

**Goal.** Score actions by reward minus a safety price times cost, because a multiplier changes which action is locally attractive. We build it in 3 steps.

In [ ]:
q_reward_e3 = np.array([2.8, 5.6]) # Reward returns for cautious and bold policies.
q_cost_e3 = np.array([0.19, 2.8]) # Cost returns for the same policies.
budget_e3 = 1.0 # Cost budget.
lambdas_e3 = np.array([0.0, 0.5, 2.0]) # Candidate safety prices.
print("reward:", q_reward_e3, "cost:", q_cost_e3) # Inspect action-level ledgers.

▶ What you'll see: bold is better on reward and worse on cost.

In [ ]:
lag_table_e3 = [] # Store Lagrangian scores for each lambda.
for lam_e3 in lambdas_e3:
    scores_e3 = q_reward_e3 - lam_e3 * (q_cost_e3 - budget_e3) # Compute L = J - lambda(C-d).
    lag_table_e3.append(scores_e3) # Save row.
    print("lambda", lam_e3, "scores", np.round(scores_e3, 3), "choice", int(np.argmax(scores_e3))) # Inspect chosen action.
lag_table_e3 = np.array(lag_table_e3) # Convert to array for plotting.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a score comparison plot.
for i_e3, name_e3 in enumerate(["cautious", "bold"]):
    plt.plot(lambdas_e3, lag_table_e3[:, i_e3], marker="o", label=name_e3) # Plot each action's score as lambda changes.
plt.xlabel("lambda"); plt.ylabel("Lagrangian score") # Label axes.
plt.title("Easy 3: multiplier changes action preference") # Title the plot.
plt.legend(); plt.show() # Display the plot.

▶ What you'll see: bold wins at low lambda, but cautious becomes better once safety is expensive enough.

👀 Takeaway: the multiplier is a learned safety price that can reverse greedy reward choices.

### Easy 4 — Dual update for the safety price

**Goal.** Update $\lambda\leftarrow[\lambda+\eta(C-d)]_+$, because constraint violation should make future cost more expensive. We build it in 3 steps.

In [ ]:
costs_seen_e4 = np.array([1.4, 1.2, 0.9, 0.7, 1.0]) # Observed policy costs across iterations.
budget_e4 = 1.0 # Safety budget.
eta_e4 = 0.6 # Dual learning rate.
lam_e4 = 0.0 # Start with no safety price.
trace_e4 = [] # Store lambda values.
print("costs seen:", costs_seen_e4) # Inspect the sequence driving dual updates.

▶ What you'll see: early costs exceed the budget, later costs fall below or equal it.

In [ ]:
for k_e4 in costs_seen_e4:
    lam_e4 = max(0.0, lam_e4 + eta_e4 * (k_e4 - budget_e4)) # Project lambda to be nonnegative.
    trace_e4.append(lam_e4) # Save new multiplier.
print("lambda trace:", np.round(trace_e4, 3)) # Inspect how the safety price responds.
assert np.all(np.array(trace_e4) >= 0) # Verify projection kept lambda nonnegative.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a dual-update chart.
plt.plot(trace_e4, marker="o", color="purple") # Plot lambda over iterations.
plt.title("Easy 4: dual ascent on lambda") # Title the plot.
plt.xlabel("iteration"); plt.ylabel("lambda") # Label axes.
plt.show() # Display the learning curve.

▶ What you'll see: lambda rises after budget violations and falls when cost is below budget.

👀 Takeaway: dual learning makes safety pressure adaptive instead of fixed by hand.

### Easy 5 — Reward-vs-cost learning trace

**Goal.** Simulate a policy becoming safer as lambda grows, because safe training should be inspected with reward and cost together. We build it in 4 steps.

In [ ]:
steps_e5 = np.arange(12) # Training iterations.
p_bold_e5 = np.linspace(0.8, 0.25, len(steps_e5)) # A toy policy gradually reduces risky action probability.
reward_e5 = 2.8 + 2.8 * p_bold_e5 # Higher bold probability gives more reward.
cost_e5 = 0.19 + 2.61 * p_bold_e5 # Higher bold probability gives more cost.
budget_e5 = 1.0 # Safety budget.
print("start reward/cost:", round(reward_e5[0], 3), round(cost_e5[0], 3)) # Inspect initial point.

▶ What you'll see: the initial policy is high reward but above budget.

In [ ]:
print("end reward/cost:", round(reward_e5[-1], 3), round(cost_e5[-1], 3)) # Inspect final point.
assert cost_e5[-1] < budget_e5 # Verify the final policy is feasible.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a time-series chart.
plt.plot(steps_e5, reward_e5, marker="o", label="reward", color="seagreen") # Plot reward over training.
plt.plot(steps_e5, cost_e5, marker="o", label="cost", color="indianred") # Plot cost over training.
plt.axhline(budget_e5, color="black", linestyle="--", label="budget") # Draw budget.
plt.title("Easy 5: inspect reward and cost together") # Title the chart.
plt.xlabel("iteration") # Label x-axis.
plt.legend(); plt.show() # Display the chart.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a reward-cost path plot.
plt.plot(cost_e5, reward_e5, marker="o", color="darkorange") # Plot the trajectory through objective space.
plt.axvline(budget_e5, color="black", linestyle="--") # Draw cost budget.
plt.xlabel("C(pi)"); plt.ylabel("J(pi)") # Label axes.
plt.title("Easy 5: reward-vs-cost path") # Title the chart.
plt.show() # Display the reward-cost path.

▶ What you'll see: the training path moves left across the budget line while reward decreases modestly.

👀 Takeaway: safe RL diagnostics should show whether lower cost is bought with acceptable reward loss.

## 🔴 Advanced

### Advanced 1 — Policy evaluation with reward and cost values

**Goal.** Run Bellman expectation backups for both reward and cost, because CMDP evaluation tracks two value functions under one policy. We build it in 4 steps.

In [ ]:
P_a1 = np.zeros((4, 2, 4)); R_a1 = np.zeros((4, 2, 4)); C_a1 = np.zeros((4, 2, 4)) # Allocate CMDP tables.
gamma_a1 = 0.9 # Discount factor.
P_a1[0, 0, 2] = 1; R_a1[0, 0, 2] = 1; C_a1[0, 0, 2] = 0.1 # start cautious.
P_a1[0, 1, 1] = 1; R_a1[0, 1, 1] = 2; C_a1[0, 1, 1] = 1.0 # start bold.
P_a1[1, 0, 3] = 1; R_a1[1, 0, 3] = 2; C_a1[1, 0, 3] = 0.2 # risky brake.
P_a1[1, 1, 3] = 1; R_a1[1, 1, 3] = 5; C_a1[1, 1, 3] = 3.0 # risky push.
P_a1[2, 0, 3] = 1; R_a1[2, 0, 3] = 2; C_a1[2, 0, 3] = 0.1 # safe finish.
P_a1[2, 1, 3] = 1; R_a1[2, 1, 3] = 3; C_a1[2, 1, 3] = 0.8 # faster safe finish.
P_a1[3, :, 3] = 1 # terminal.
print("transition rows valid:", np.allclose(P_a1.sum(axis=2), 1.0)) # Verify probabilities.

▶ What you'll see: every state-action row has total transition probability one.

In [ ]:
pi_a1 = np.array([[0.7, 0.3], [0.8, 0.2], [0.6, 0.4], [1.0, 0.0]]) # Fixed stochastic policy.
Vr_a1 = np.zeros(4); Vc_a1 = np.zeros(4) # Initialize reward and cost values.
print("policy row sums:", pi_a1.sum(axis=1)) # Inspect policy validity.

In [ ]:
for _ in range(60):
    old_r_a1, old_c_a1 = Vr_a1.copy(), Vc_a1.copy() # Use previous iteration values for synchronous backups.
    for s_a1 in range(4):
        Vr_a1[s_a1] = sum(pi_a1[s_a1, a_a1] * np.sum(P_a1[s_a1, a_a1] * (R_a1[s_a1, a_a1] + gamma_a1 * old_r_a1)) for a_a1 in range(2)) # Reward Bellman expectation.
        Vc_a1[s_a1] = sum(pi_a1[s_a1, a_a1] * np.sum(P_a1[s_a1, a_a1] * (C_a1[s_a1, a_a1] + gamma_a1 * old_c_a1)) for a_a1 in range(2)) # Cost Bellman expectation.
print("V_reward:", np.round(Vr_a1, 3)) # Inspect reward values.
print("V_cost:", np.round(Vc_a1, 3)) # Inspect cost values.
assert round(Vr_a1[0], 3) == 3.514 and round(Vc_a1[0], 3) == 0.815 # Verify start values.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3)) # Create side-by-side value charts.
ax[0].bar(["start", "risky", "safe", "done"], Vr_a1, color="seagreen") # Plot reward values.
ax[0].set_title("Advanced 1: reward value") # Title reward plot.
ax[0].tick_params(axis="x", rotation=20) # Rotate labels.
ax[1].bar(["start", "risky", "safe", "done"], Vc_a1, color="indianred") # Plot cost values.
ax[1].set_title("Advanced 1: cost value") # Title cost plot.
ax[1].tick_params(axis="x", rotation=20) # Rotate labels.
plt.tight_layout(); plt.show() # Display plots.

▶ What you'll see: risky state has both high reward value and high cost value.

👀 Takeaway: CMDP evaluation duplicates the Bellman machinery for the cost ledger.

### Advanced 2 — Tabular constrained policy improvement

**Goal.** Improve a policy by selecting actions with high reward advantage after subtracting a cost price, because the Lagrangian changes the greedy backup. We build it in 4 steps.

In [ ]:
Vr_a2 = Vr_a1.copy(); Vc_a2 = Vc_a1.copy() # Reuse evaluated reward and cost values from Advanced 1.
lam_a2 = 1.5 # Set a safety price for improvement.
Q_lag_a2 = np.zeros((4, 2)) # Store Lagrangian action values.
print("lambda:", lam_a2) # Inspect multiplier.

▶ What you'll see: lambda is the cost price used during action selection.

In [ ]:
for s_a2 in range(4):
    for a_a2 in range(2):
        qr_a2 = np.sum(P_a1[s_a2, a_a2] * (R_a1[s_a2, a_a2] + gamma_a1 * Vr_a2)) # Reward action value.
        qc_a2 = np.sum(P_a1[s_a2, a_a2] * (C_a1[s_a2, a_a2] + gamma_a1 * Vc_a2)) # Cost action value.
        Q_lag_a2[s_a2, a_a2] = qr_a2 - lam_a2 * qc_a2 # Safety-priced action value.
print("Lagrangian Q:\n", np.round(Q_lag_a2, 3)) # Inspect priced action values.

In [ ]:
new_actions_a2 = np.argmax(Q_lag_a2, axis=1) # Greedy improvement under priced values.
print("greedy actions:", new_actions_a2) # Inspect selected action per state.
assert new_actions_a2[0] == 0 and new_actions_a2[1] == 0 # Verify high lambda prefers safer early/risky actions.

In [ ]:
plt.figure(figsize=(5, 3)) # Create action-value heatmap.
plt.imshow(Q_lag_a2[:3], cmap="viridis", aspect="auto") # Show nonterminal action scores.
plt.colorbar(label="Q_R - lambda Q_C") # Add colorbar.
plt.xticks([0, 1], ["cautious", "bold"]) # Label actions.
plt.yticks([0, 1, 2], ["start", "risky", "safe"]) # Label states.
plt.title("Advanced 2: safety-priced Q values") # Title heatmap.
plt.show() # Display heatmap.

▶ What you'll see: the high-cost bold action in the risky state is darkened by the multiplier.

👀 Takeaway: Lagrangian improvement greedifies reward after subtracting priced expected cost.

### Advanced 3 — Primal-dual training of one policy logit

**Goal.** Learn a bold-action probability and a multiplier together, because constrained optimization alternates policy improvement with budget enforcement. We build it in 4 steps.

In [ ]:
logit_a3 = 0.0 # Learnable start-state bold logit.
lam_a3 = 0.0 # Learnable nonnegative multiplier.
eta_pi_a3 = 0.8; eta_lam_a3 = 0.8 # Learning rates for primal and dual steps.
budget_a3 = 0.9 # Cost budget.
trace_a3 = [] # Store p, reward, cost, lambda.
print("initial p_bold:", round(1 / (1 + np.exp(-logit_a3)), 3)) # Inspect initial probability.

▶ What you'll see: the initial policy chooses bold half the time at the start.

In [ ]:
def eval_start_a3(p_bold):
    pi_tmp_a3 = np.array([[1 - p_bold, p_bold], [0.8, 0.2], [0.6, 0.4], [1.0, 0.0]]) # Policy with one variable row.
    vr_a3 = np.zeros(4); vc_a3 = np.zeros(4) # Initialize values.
    for _ in range(50):
        or_a3, oc_a3 = vr_a3.copy(), vc_a3.copy() # Synchronous old values.
        for s_a3 in range(4):
            vr_a3[s_a3] = sum(pi_tmp_a3[s_a3, a_a3] * np.sum(P_a1[s_a3, a_a3] * (R_a1[s_a3, a_a3] + gamma_a1 * or_a3)) for a_a3 in range(2)) # Reward backup.
            vc_a3[s_a3] = sum(pi_tmp_a3[s_a3, a_a3] * np.sum(P_a1[s_a3, a_a3] * (C_a1[s_a3, a_a3] + gamma_a1 * oc_a3)) for a_a3 in range(2)) # Cost backup.
    return vr_a3[0], vc_a3[0]
print("eval at p=0.5:", np.round(eval_start_a3(0.5), 3)) # Inspect one evaluation.

In [ ]:
for _ in range(35):
    p_a3 = 1 / (1 + np.exp(-logit_a3)) # Convert logit to probability.
    J_a3, K_a3 = eval_start_a3(p_a3) # Evaluate current policy.
    trace_a3.append([p_a3, J_a3, K_a3, lam_a3]) # Store diagnostics.
    eps_a3 = 1e-3 # Finite-difference step.
    p_plus_a3 = 1 / (1 + np.exp(-(logit_a3 + eps_a3))) # Probability after positive logit perturbation.
    p_minus_a3 = 1 / (1 + np.exp(-(logit_a3 - eps_a3))) # Probability after negative logit perturbation.
    Jp_a3, Kp_a3 = eval_start_a3(p_plus_a3); Jm_a3, Km_a3 = eval_start_a3(p_minus_a3) # Evaluate both perturbations.
    grad_a3 = ((Jp_a3 - lam_a3 * (Kp_a3 - budget_a3)) - (Jm_a3 - lam_a3 * (Km_a3 - budget_a3))) / (2 * eps_a3) # Lagrangian gradient.
    logit_a3 += eta_pi_a3 * grad_a3 # Primal ascent on policy logit.
    lam_a3 = max(0.0, lam_a3 + eta_lam_a3 * (K_a3 - budget_a3)) # Dual ascent on multiplier.
trace_a3 = np.array(trace_a3) # Convert diagnostics to array.
print("final row:", np.round(trace_a3[-1], 3)) # Inspect final p, reward, cost, lambda.
assert trace_a3[-1, 2] < 1.05 # Verify cost was pulled near the budget.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3)) # Create training diagnostics.
ax[0].plot(trace_a3[:, 1], label="reward", color="seagreen") # Plot reward.
ax[0].plot(trace_a3[:, 2], label="cost", color="indianred") # Plot cost.
ax[0].axhline(budget_a3, color="black", linestyle="--", label="budget") # Draw budget.
ax[0].set_title("Advanced 3: reward/cost") # Title first panel.
ax[0].legend() # Show legend.
ax[1].plot(trace_a3[:, 0], label="p(bold)", color="purple") # Plot policy probability.
ax[1].plot(trace_a3[:, 3], label="lambda", color="gray") # Plot multiplier.
ax[1].set_title("Advanced 3: primal/dual") # Title second panel.
ax[1].legend() # Show legend.
plt.tight_layout(); plt.show() # Display panels.

▶ What you'll see: cost initially rises with reward, then the multiplier grows and pushes bold probability downward.

👀 Takeaway: primal-dual learning is a feedback loop between reward-seeking and constraint correction.

### Advanced 4 — Cost-aware Q-learning on a toy table

**Goal.** Learn action values for the shaped reward $r-\lambda c$, because a fixed multiplier converts a CMDP into an ordinary tabular control problem for one training phase. We build it in 4 steps.

In [ ]:
Q_a4 = np.zeros((3, 2)) # Q table for states start, mid, done and two actions.
gamma_a4 = 0.9; alpha_a4 = 0.5; lam_a4 = 2.0 # Set learning hyperparameters and safety price.
transitions_a4 = [(0, 0, 1, 0.1, 1), (0, 1, 2, 1.0, 1), (1, 0, 2, 0.1, 2), (1, 1, 4, 2.0, 2)] # (s,a,r,c,next).
print("initial Q:\n", Q_a4) # Inspect blank table.

▶ What you'll see: all shaped action values start at zero.

In [ ]:
for epoch_a4 in range(20):
    for s_a4, a_a4, r_a4, c_a4, sp_a4 in transitions_a4:
        shaped_a4 = r_a4 - lam_a4 * c_a4 # Convert reward/cost into one Lagrangian reward.
        target_a4 = shaped_a4 + gamma_a4 * np.max(Q_a4[sp_a4]) # Q-learning target.
        Q_a4[s_a4, a_a4] += alpha_a4 * (target_a4 - Q_a4[s_a4, a_a4]) # Update selected cell.
print("learned shaped Q:\n", np.round(Q_a4, 3)) # Inspect learned action values.
assert Q_a4[0, 0] > Q_a4[0, 1] # Verify safety price makes cautious start better.

In [ ]:
greedy_a4 = np.argmax(Q_a4, axis=1) # Choose greedy action under shaped values.
print("greedy shaped actions:", greedy_a4) # Inspect policy implied by fixed lambda.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a Q heatmap.
plt.imshow(Q_a4, cmap="viridis", aspect="auto") # Visualize shaped action values.
plt.colorbar(label="Q for r - lambda c") # Add colorbar.
plt.xticks([0, 1], ["cautious", "bold"]) # Label actions.
plt.yticks([0, 1, 2], ["start", "mid", "done"]) # Label states.
plt.title("Advanced 4: cost-aware Q-learning") # Title heatmap.
plt.show() # Display heatmap.

▶ What you'll see: the shaped value table prefers cautious actions when cost is priced at lambda = 2.

👀 Takeaway: a fixed multiplier lets ordinary Q-learning optimize a safety-priced reward, but the multiplier still needs separate tuning.

### Advanced 5 — Offline support check for a proposed policy

**Goal.** Compare behavior-data action frequencies with a proposed safe policy, because off-policy cost estimates are unreliable when the dataset rarely sampled the evaluated action. We build it in 4 steps.

In [ ]:
counts_a5 = np.array([[90, 10], [80, 20], [60, 40]]) # Offline action counts for three states.
behavior_a5 = counts_a5 / counts_a5.sum(axis=1, keepdims=True) # Convert counts to behavior probabilities.
proposed_a5 = np.array([[0.4, 0.6], [0.3, 0.7], [0.5, 0.5]]) # Proposed policy to evaluate.
print("behavior:\n", np.round(behavior_a5, 2)) # Inspect data support.
print("proposed:\n", proposed_a5) # Inspect evaluation policy.

▶ What you'll see: the proposed policy puts much more probability on action 1 than the data did.

In [ ]:
ratio_a5 = np.divide(proposed_a5, behavior_a5, out=np.zeros_like(proposed_a5), where=behavior_a5 > 0) # Importance ratios by state/action.
max_ratio_a5 = float(np.max(ratio_a5)) # Largest extrapolation factor.
print("ratios:\n", np.round(ratio_a5, 2)) # Inspect support mismatch.
print("max ratio:", round(max_ratio_a5, 2)) # Inspect worst case.
assert round(max_ratio_a5, 2) == 6.0 # Verify state 0 action 1 is 0.6 / 0.1.

In [ ]:
flags_a5 = ratio_a5 > 3.0 # Flag proposed probabilities far above behavior support.
print("support flags:\n", flags_a5.astype(int)) # Inspect risky extrapolation cells.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 3)) # Create side-by-side heatmaps.
ax[0].imshow(behavior_a5, cmap="Blues", vmin=0, vmax=1, aspect="auto") # Behavior policy support.
ax[0].set_title("behavior data") # Title first panel.
ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1, 2]) # Set ticks.
ax[1].imshow(proposed_a5, cmap="Oranges", vmin=0, vmax=1, aspect="auto") # Proposed policy mass.
ax[1].set_title("proposed policy") # Title second panel.
ax[1].set_xticks([0, 1]); ax[1].set_yticks([0, 1, 2]) # Set ticks.
plt.suptitle("Advanced 5: policy support mismatch") # Overall title.
plt.tight_layout(); plt.show() # Display heatmaps.

▶ What you'll see: proposed action-1 probabilities are bright where behavior action-1 support is dark.

👀 Takeaway: offline safe RL must check support, because unseen risky actions can have underestimated cost.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Constrained RL makes safety a mathematical budget, not an afterthought.

Reinforcement learning is where a prediction changes what data arrives next. Probability supplies transition probabilities and expectations; optimization supplies iterative improvement. These notebooks keep the environments tiny and CPU-only while implementing the real decision rule. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1135
random.seed(SEED)
np.random.seed(SEED)

ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = ["up", "right", "down", "left"]


@dataclass
class GridEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    hazards: set
    traps: set
    slip: float
    wind: float
    budget: float
    max_steps: int
    reward_goal: float = 5.0
    reward_step: float = -0.05
    cost_hazard: float = 1.0

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return 4

    def state_to_pos(self, state):
        row = state // self.width
        col = state % self.width
        return (row, col)

    def pos_to_state(self, pos):
        return pos[0] * self.width + pos[1]

    def in_bounds(self, pos):
        row, col = pos
        return 0 <= row < self.height and 0 <= col < self.width

    def start_state(self):
        return self.pos_to_state(self.start)

    def goal_state(self):
        return self.pos_to_state(self.goal)

    def transition(self, state, action, slip_override=None):
        slip = self.slip if slip_override is None else slip_override
        probs = []
        primary = self._move(state, action)
        probs.append((1.0 - slip, primary))
        side_actions = [(action + 1) % 4, (action - 1) % 4]
        for side in side_actions:
            probs.append((slip / 2.0, self._move(state, side)))
        merged = {}
        for prob, next_state in probs:
            merged[next_state] = merged.get(next_state, 0.0) + prob
        return list(merged.items())

    def _move(self, state, action):
        if state == self.goal_state():
            return state
        pos = np.array(self.state_to_pos(state))
        next_pos = tuple(pos + ACTIONS[action])
        if self.wind > 0 and action == 1:
            next_pos = (max(0, next_pos[0] - 1), next_pos[1])
        if not self.in_bounds(next_pos):
            next_pos = tuple(pos)
        return self.pos_to_state(next_pos)

    def reward_cost_done(self, state, action, next_state):
        pos = self.state_to_pos(next_state)
        done = next_state == self.goal_state()
        reward = self.reward_goal if done else self.reward_step
        cost = self.cost_hazard if pos in self.hazards or pos in self.traps else 0.0
        if pos in self.traps:
            reward -= 1.0
        return reward, cost, done


def make_f12_ladder():
    return [
        GridEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), {(0, 1)}, set(), 0.00, 0.0, 1.0, 3),
        GridEnv("D2 slippery 3-state hazards", 1, 3, (0, 0), (0, 2), {(0, 1)}, set(), 0.10, 0.0, 1.0, 6),
        GridEnv("D3 4x4 gridworld hazards", 4, 4, (3, 0), (0, 3), {(2, 1), (1, 2)}, set(), 0.08, 0.0, 1.2, 18),
        GridEnv("D4 stochastic windy grid", 5, 5, (4, 0), (0, 4), {(3, 1), (2, 2), (1, 3)}, set(), 0.15, 0.25, 1.4, 28),
        GridEnv("D5 sparse grid with traps", 6, 6, (5, 0), (0, 5), {(4, 1), (3, 2), (2, 3)}, {(1, 4), (4, 4)}, 0.18, 0.30, 1.5, 40),
    ]


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    for step, reward in enumerate(rewards):
        total += (gamma ** step) * reward
    return total


def value_iteration(env, gamma=0.9, penalty=0.0, slip_override=None, iterations=80):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            if state == env.goal_state():
                continue
            action_scores = []
            for action in range(env.n_actions):
                score = 0.0
                for prob, next_state in env.transition(state, action, slip_override):
                    reward, cost, done = env.reward_cost_done(state, action, next_state)
                    bootstrap = 0.0 if done else gamma * values[next_state]
                    score += prob * (reward - penalty * cost + bootstrap)
                action_scores.append(score)
            new_values[state] = max(action_scores)
            q_values[state] = action_scores
        values = new_values
    policy = np.argmax(q_values, axis=1)
    return values, q_values, policy


def evaluate_policy(env, policy, gamma=0.9, episodes=40, seed=0, slip_override=None):
    rng = np.random.default_rng(seed)
    returns = []
    costs = []
    wins = []
    paths = []
    for episode in range(episodes):
        state = env.start_state()
        rewards = []
        cost_values = []
        path = [state]
        for step in range(env.max_steps):
            action = int(policy[state])
            transitions = env.transition(state, action, slip_override)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            rewards.append(reward)
            cost_values.append(cost)
            state = next_state
            path.append(state)
            if done:
                break
        returns.append(discounted_return(rewards, gamma))
        costs.append(sum(cost_values))
        wins.append(float(state == env.goal_state()))
        paths.append(path)
    return {
        "return": float(np.mean(returns)),
        "cost": float(np.mean(costs)),
        "win_rate": float(np.mean(wins)),
        "path": paths[0],
    }



def preview_ladder(ladder):
    rows = []
    for idx, env in enumerate(ladder, start=1):
        rows.append({
            "rung": f"D{idx}",
            "name": env.name,
            "shape": (env.height, env.width),
            "states": env.n_states,
            "slip": env.slip,
            "budget": env.budget,
            "hazards": len(env.hazards) + len(env.traps),
        })
    return rows


def plot_env_panel(ax, env, values, title):
    image = np.asarray(values).reshape(env.height, env.width)
    ax.imshow(image, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for hazard in env.hazards:
        ax.text(hazard[1], hazard[0], "H", color="white", ha="center", va="center")
    for trap in env.traps:
        ax.text(trap[1], trap[0], "T", color="red", ha="center", va="center")
    ax.text(env.start[1], env.start[0], "S", color="white", ha="center", va="center")
    ax.text(env.goal[1], env.goal[0], "G", color="white", ha="center", va="center")


## The concept, built once (D1)

The lesson objective is $$\max_\pi J(\pi)\quad \text{subject to}\quad C(\pi)\le d$$. First verify the shared discounted-return, bootstrap, softmax, and UCB numbers exactly.

In [ ]:

rewards = [1, 0, 2]
gamma = 0.9
G = discounted_return(rewards, gamma)
y = 1 + gamma * 0.8
q_new = 0.4 + 0.5 * (y - 0.4)
probs = softmax([1, 0])
expected_reward = probs[0] * 2 + probs[1] * 0
ucb = 0.55 + math.sqrt(2 * math.log(20) / 5)
print("G", round(G, 3))
print("target", round(y, 3))
print("Q_new", round(q_new, 3))
print("policy", np.round(probs, 3))
print("expected reward", round(float(expected_reward), 3))
print("UCB", round(ucb, 3))
assert round(G, 3) == 2.620
assert round(y, 3) == 1.720
assert round(q_new, 3) == 1.060
assert round(float(probs[0]), 3) == 0.731
assert round(float(probs[1]), 3) == 0.269
assert round(float(expected_reward), 3) == 1.462
assert round(ucb, 3) == 1.645


Now use the same object on D1: evaluate reward return, cost return, and a Lagrangian multiplier that penalizes expected cost above the budget.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    def constrained_policy_eval(env, policy, gamma=0.9, seed=7):
        metrics = evaluate_policy(env, policy, gamma=gamma, episodes=60, seed=seed)
        violation = max(0.0, metrics["cost"] - env.budget)
        metrics["violation"] = violation
        metrics["constrained_return"] = metrics["return"] - 2.0 * violation
        return metrics


    def lagrangian_constrained_policy(env, gamma=0.9, iterations=9, lr=0.55):
        lam = 0.0
        history = []
        policy = np.zeros(env.n_states, dtype=int)
        for iteration in range(iterations):
            values, q_values, policy = value_iteration(env, gamma=gamma, penalty=lam, iterations=90)
            metrics = constrained_policy_eval(env, policy, gamma=gamma, seed=SEED + iteration)
            lam = max(0.0, lam + lr * (metrics["cost"] - env.budget))
            metrics["lambda"] = lam
            metrics["iteration"] = iteration
            history.append(metrics)
        return policy, values, history


    def reward_only_policy(env, gamma=0.9):
        values, q_values, policy = value_iteration(env, gamma=gamma, penalty=0.0, iterations=90)
        return policy, values

    env = make_f12_ladder()[0]
    policy, values, history = lagrangian_constrained_policy(env)
    metrics = constrained_policy_eval(env, policy)
    print(metrics)
    assert metrics["cost"] <= env.budget + 0.6
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## The dataset ladder

Family F12 uses a D1–D5 sequential-decision ladder inline: a two-state chain, slippery chain, 4x4 grid, windy grid, and sparse trap grid.

In [ ]:
ladder = make_f12_ladder()
for row in preview_ladder(ladder):
    print(row)
print("sample D5 policy grid shape", (ladder[-1].height, ladder[-1].width))

## Run the SAME method across D1-D5

Apply the same method to every rung and collect the plan metric.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    results = []
    artifacts = []
    for index, env in enumerate(ladder, start=1):
        policy, values, history = lagrangian_constrained_policy(env)
        metrics = constrained_policy_eval(env, policy, seed=SEED + index)
        metrics["rung"] = f"D{index}"
        metrics["name"] = env.name
        results.append(metrics)
        artifacts.append((env, values, policy, history))
    print("rung | return | cost | violation | constrained_return | win_rate")
    for row in results:
        print(row["rung"], round(row["return"], 3), round(row["cost"], 3), round(row["violation"], 3), round(row["constrained_return"], 3), round(row["win_rate"], 3))
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for col, (env, values, policy, history) in enumerate(artifacts):
        plot_env_panel(axes[0, col], env, values, f"D{col + 1} value")
        cost_curve = [item["cost"] for item in history]
        axes[1, col].plot(cost_curve, label="cost")
        axes[1, col].axhline(env.budget, color="red", linestyle="--", label="budget")
        axes[1, col].set_title(f"D{col + 1} cost")
        axes[1, col].set_xlabel("iteration")
    axes[1, 0].legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    xs = np.arange(1, 6)
    plt.plot(xs, [row["constrained_return"] for row in results], marker="o", label="constrained return")
    plt.plot(xs, [row["cost"] for row in results], marker="s", label="cost")
    plt.xticks(xs, [row["rung"] for row in results])
    plt.xlabel("rung")
    plt.ylabel("metric")
    plt.title("Return / constraint cost across D1-D5")
    plt.legend()
    plt.show()
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on D5: confusing reward with return

A reward-only policy may collect faster goal reward while exceeding the cost budget. The fix is the Lagrangian policy plus a feasible-cost check.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    env = ladder[-1]
    wrong_policy, wrong_values = reward_only_policy(env)
    wrong = constrained_policy_eval(env, wrong_policy, seed=99)
    fixed_policy, fixed_values, fixed_history = lagrangian_constrained_policy(env)
    fixed = constrained_policy_eval(env, fixed_policy, seed=99)
    print("wrong reward-only", wrong)
    print("fixed constrained", fixed)
    print("violation improvement", round(wrong["violation"] - fixed["violation"], 3))
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Evaluate it + Practice

- Compare the reported constrained return / constraint violation / win-rate against a no-skill baseline such as a random or immediate-reward policy.
- Sanity check that the D1 result matches the exact lesson arithmetic before trusting harder rungs.
- Ablate the key idea, such as removing constraints, randomization, return conditioning, population replay, or search.
- Watch failure signals: budget violations, transfer collapse, unsupported target returns, non-stationary opponents, or shallow reward chasing.

Practice:
1. Change the discount from 0.9 to 0.8 and predict which rung changes most.


2. Add one hazard or trap to D4 and rerun only the small table, not a long training job.

3. Replace the no-skill baseline with a hand-written safe policy and compare the metric.